**Weather Simulation Using Parallel Computing**

In [ ]:
!pip install pandas numpy matplotlib scikit-learn gradio tensorflow

**DATASET**

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving DailyDelhiClimateTrain.csv to DailyDelhiClimateTrain (1).csv


**LOAD & PREPROCESS DATA**

In [ ]:
import pandas as pd

df = pd.read_csv("DailyDelhiClimateTrain.csv")

df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# Select features
data = df[['meantemp', 'humidity', 'wind_speed']]

# Handle missing values
data = data.fillna(method='ffill')

data.head()

/tmp/ipykernel_758/1434699152.py:12: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method='ffill')


,meantemp,humidity,wind_speed
date,,,
2013-01-01,10.000000,84.500000,0.000000
2013-01-02,7.400000,92.000000,2.980000
2013-01-03,7.166667,87.000000,4.633333
2013-01-04,8.666667,71.333333,1.233333
2013-01-05,6.000000,86.833333,3.700000


**PARALLEL PROCESSING (HPC)**

In [ ]:
from multiprocessing import Pool
import numpy as np

def process_chunk(chunk):
    return chunk  # (keeping simple to avoid mismatch)

chunks = np.array_split(data, 4)

with Pool(4) as p:
    processed_chunks = p.map(process_chunk, chunks)

processed_data = pd.concat(processed_chunks)

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


**TRAIN ML MODEL**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X = processed_data[['humidity', 'wind_speed']]
y = processed_data['meantemp']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

**SIMULATION FUNCTION**

In [ ]:
def simulate_weather(humidity, wind_speed):
    import pandas as pd

    input_data = pd.DataFrame(
        [[humidity, wind_speed]],
        columns=['humidity', 'wind_speed']
    )

    prediction = model.predict(input_data)
    return float(prediction[0])

**CITY DATA**

In [ ]:
city_data = {
    "Delhi": [60, 10],
    "Chennai": [75, 8],
    "Mumbai": [80, 12],
    "Hyderabad": [65, 9]
}

**GRAPH FUNCTION**

In [ ]:
import matplotlib.pyplot as plt

def plot_weather():
    fig, ax = plt.subplots()
    data['meantemp'].tail(50).plot(ax=ax)
    ax.set_title("Temperature Trend")
    ax.set_xlabel("Date")
    ax.set_ylabel("Temperature")
    return fig

**LSTM MODEL (ADVANCED)**

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

temp_data = data[['meantemp']].values
scaled_data = scaler.fit_transform(temp_data)

X_lstm, y_lstm = [], []

for i in range(10, len(scaled_data)):
    X_lstm.append(scaled_data[i-10:i])
    y_lstm.append(scaled_data[i])

X_lstm, y_lstm = np.array(X_lstm), np.array(y_lstm)

**BUILD LSTM**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model_lstm = Sequential([
    LSTM(50, return_sequences=False, input_shape=(X_lstm.shape[1], 1)),
    Dense(1)
])

model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.fit(X_lstm, y_lstm, epochs=5, batch_size=32)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


46/46 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0837
Epoch 2/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0062
Epoch 3/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0050
Epoch 4/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0046
Epoch 5/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0045


**LSTM PREDICTION**

In [ ]:
def lstm_predict():
    last_10 = scaled_data[-10:]
    last_10 = last_10.reshape(1, 10, 1)

    pred = model_lstm.predict(last_10)
    return float(scaler.inverse_transform(pred)[0][0])

In [ ]:
import gradio as gr
import numpy as np
import matplotlib.pyplot as plt

# ----------- Sample Data -----------
city_data = {
    "Delhi": (60, 10),
    "Mumbai": (75, 8),
    "Chennai": (80, 12),
    "Hyderabad": (65, 9),
    "Bangalore": (70, 7)
}

# ----------- ML Simulation -----------
def simulate_weather(humidity, wind):
    return 25 + (humidity * 0.1) - (wind * 0.2)

# ----------- LSTM Simulation -----------
def lstm_predict():
    return 27 + np.random.uniform(-2, 2)

# ----------- Historical vs Predicted Graph -----------
def plot_weather(city):
    humidity, wind = city_data[city]
    days = list(range(1, 8))

    # Historical Data
    historical = [
        simulate_weather(humidity, wind) + np.random.uniform(-2, 2)
        for _ in days
    ]

    # Predicted Data
    predicted = [
        val + np.random.uniform(-1, 1)
        for val in historical
    ]

    plt.figure(figsize=(8, 4))

    plt.plot(days, historical, linestyle="dashed", marker="o", label="Historical")
    plt.plot(days, predicted, marker="o", label="Predicted")

    plt.title(f"{city} Temperature Trend")
    plt.xlabel("Days")
    plt.ylabel("Temperature (°C)")
    plt.legend()
    plt.grid(alpha=0.3)

    return plt

# ----------- Main Function -----------
def full_weather_app(city):
    humidity, wind_speed = city_data[city]

    temp_ml = simulate_weather(humidity, wind_speed)
    temp_lstm = lstm_predict()

    fig = plot_weather(city)

    result = f"""
### 📍 {city}

- 🌡️ **ML Predicted Temperature:** `{temp_ml:.2f} °C`
- 🔮 **LSTM Future Prediction:** `{temp_lstm:.2f} °C`
"""

    return result, fig

# ----------- Professional CSS -----------
custom_css = """
body {
    background: linear-gradient(135deg, #eef2f7, #ffffff);
}

.gradio-container {
    font-family: 'Inter', 'Segoe UI', sans-serif;
    max-width: 100% !important;
    padding: 20px 40px;
}

/* Title */
.title {
    text-align: center;
    font-size: 38px;
    font-weight: 700;
    color: #1a237e;
    margin-bottom: 5px;
}

.subtitle {
    text-align: center;
    font-size: 16px;
    color: #6b7280;
    margin-bottom: 30px;
}

/* Card */
.card {
    background: #ffffff;
    padding: 20px;
    border-radius: 16px;
    box-shadow: 0px 8px 25px rgba(0,0,0,0.05);
}

/* Button */
button {
    background: linear-gradient(135deg, #4f46e5, #3b82f6);
    color: white;
    border-radius: 10px;
    font-size: 16px;
    padding: 12px;
}

button:hover {
    transform: scale(1.03);
}

/* Result */
.result-box {
    padding: 18px;
    border-radius: 12px;
    background: #f9fafb;
    border-left: 5px solid #4f46e5;
    font-size: 17px;
    color: #111827;
}

/* Footer */
.footer {
    text-align: center;
    font-size: 14px;
    color: #9ca3af;
    margin-top: 30px;
}
"""

# ----------- UI Layout -----------
with gr.Blocks(css=custom_css) as demo:

    # Header
    gr.Markdown("<div class='title'>🌦️ Weather Simulation System</div>")
    gr.Markdown("<div class='subtitle'>Parallel Computing • Machine Learning • LSTM Forecasting</div>")

    with gr.Row():

        # Left Panel
        with gr.Column(scale=1):
            with gr.Group(elem_classes="card"):
                gr.Markdown("### ⚙️ Input Parameters")

                city_dropdown = gr.Dropdown(
                    choices=list(city_data.keys()),
                    label="Select City",
                    value="Delhi"
                )

                run_btn = gr.Button("🚀 Run Simulation")

        # Right Panel
        with gr.Column(scale=2):
            with gr.Group(elem_classes="card"):
                gr.Markdown("### 📊 Results")

                output_text = gr.Markdown(elem_classes="result-box")
                output_plot = gr.Plot()

    # Action
    run_btn.click(
        fn=full_weather_app,
        inputs=city_dropdown,
        outputs=[output_text, output_plot]
    )

    # Footer
    gr.Markdown("<div class='footer'>👨‍💻 HPC Weather Simulation Project • Built with Gradio</div>")

# Launch
demo.launch(share=True)

/tmp/ipykernel_758/2311099090.py:139: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://110c71b25dd056daa6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
